# Lab RFVI — Content-Based Image Retrieval (Flowers)
**Companion to the handout `tp_cbir_handout.pdf`.**

Fill in the cells marked `# TODO`. Do not change the signatures of the functions:
the handout and the grading script rely on them.

**Workflow.**
1. Run `[Setup]` once (installs, seeds, globals).
2. Run `[Data]` to load the gallery and queries.
3. Part 1: load the four backbones and extract features.
4. Part 2: implement the metrics, fill Table 2 of the handout.
5. Part 3: fine-tune (optional, GPU recommended).
6. Part 4: compare, t-SNE, confusion matrix, visual grids.
7. Part 5: FAISS trade-off, recall/time curve.
8. Answer the six questions in the last cell.

In [ ]:
# [Setup]
import os, time, copy
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRETRAINED  = True
FINE_TUNE   = True
EPOCHS      = 15 if DEVICE == "cuda" else 3
NUM_WORKERS = 0 if os.name == "nt" else 2
IMG_SIZE    = 128
print("device:", DEVICE, "| fine-tune:", FINE_TUNE, "| epochs:", EPOCHS)

In [ ]:
# [Data]
SPLIT_DIR = Path("../datasets/split/flowers")
CLASS_NAMES = ["daisy", "dandelion", "rose", "sunflower", "tulip"]
NUM_CLASSES = len(CLASS_NAMES)
assert SPLIT_DIR.exists(), "data/split/ not found: run Lab 3 first"

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")
all_df   = pd.concat([train_df, val_df, test_df], ignore_index=True)

def load_image(path, size=(IMG_SIZE, IMG_SIZE)):
    with Image.open(path) as img:
        return np.array(img.convert("RGB").resize(size))

X_db = np.array([load_image(p) for p in all_df["filepath"]])
y_db = all_df["label"].to_numpy()
n_db = len(X_db)
print("Gallery :", n_db, "images | classes:", np.bincount(y_db))

rng = np.random.default_rng(SEED)
test_pos = np.arange(len(train_df) + len(val_df), n_db)
q_pos = rng.choice(test_pos, size=min(100, len(test_pos)), replace=False)
print("Queries :", len(q_pos))

## Part 1 — Backbones and feature extraction

In [ ]:
class Gallery(Dataset):
    def __init__(self, X, tf): self.X, self.tf = X, tf
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.tf(Image.fromarray(self.X[i]))

@torch.no_grad()
def embed_backbone(encoder, X, bs=64):
    encoder.eval()
    loader = DataLoader(Gallery(X, tf_eval), batch_size=bs,
                        shuffle=False, num_workers=NUM_WORKERS)
    out = []
    for batch in loader:
        batch = batch.to(DEVICE)
        out.append(encoder(batch).cpu())
    return torch.cat(out, dim=0).numpy()


In [ ]:
# [Part 1.2] External SSL checkpoints (SimCLR, MoCo, BYOL) from Hugging Face
# Robust loader: format-aware extraction, prefix stripping, strict verification.
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

SSL_MODELS = {
    "SimCLR": {
        "repo_id": "DM-Diaz/SimCLR-ResNet18-ImageNet1K",
        "filename": "checkpoint_100-resnet18-simclr-imagenet1k.ckpt",
        "architecture": "resnet18",
        "format": "pytorch",
    },
 
    "BYOL": {
        "repo_id": "niels-c/resnet50_byol_linhead",
        "filename": "resnet50_byol_linhead.safetensors",
        "architecture": "resnet50",
        "format": "safetensors",
    },
}

_SEQ2NAME = {"0": "conv1", "1": "bn1", "4": "layer1",
             "5": "layer2", "6": "layer3", "7": "layer4"}


def download_checkpoint(config):
    print(f"\nDownloading: {config['repo_id']}")
    path = hf_hub_download(repo_id=config["repo_id"],
                           filename=config["filename"])
    print(f"Downloaded: {path}")
    return path


def extract_state_dict(path, file_format):
    if file_format == "safetensors":
        state_dict = load_file(path, device="cpu")
    elif file_format == "pytorch":
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        if not isinstance(checkpoint, dict):
            raise RuntimeError("Invalid checkpoint format")
        state_dict = checkpoint
        for key in ("state_dict", "model_state_dict", "model"):
            if key in state_dict and isinstance(state_dict[key], dict):
                state_dict = state_dict[key]
                break
    else:
        raise ValueError(f"Unknown format: {file_format}")
    return state_dict


def convert_keys(state_dict, prefix):
    converted = {}
    for key, value in state_dict.items():
        if not isinstance(value, torch.Tensor):
            continue
        if not key.startswith(prefix):
            continue
        key = key[len(prefix):]
        head, separator, rest = key.partition(".")
        if separator and head in _SEQ2NAME:
            key = f"{_SEQ2NAME[head]}.{rest}"
        converted[key] = value
    return converted


def load_ssl_model(config):
    repo_id = config["repo_id"]
    architecture = config["architecture"]
    print(f"\n{'=' * 65}")
    print(f"Loading {repo_id}")
    print(f"Architecture: {architecture}")
    print(f"{'=' * 65}")

    path = download_checkpoint(config)
    state_dict = extract_state_dict(path, config["format"])

    if architecture == "resnet18":
        model = torchvision.models.resnet18(weights=None)
    elif architecture == "resnet50":
        model = torchvision.models.resnet50(weights=None)
    else:
        raise ValueError(f"Unsupported architecture: {architecture}")

    model.fc = nn.Identity()
    target = model.state_dict()
    target_keys = set(target.keys())

    prefixes = [
        "backbone.", "module.backbone.", "model.backbone.",
        "encoder_q.", "module.encoder_q.", "online_encoder.",
        "module.", "model.", "",
    ]

    best = None
    for prefix in prefixes:
        converted = convert_keys(state_dict, prefix)
        compatible = {
            key: value for key, value in converted.items()
            if key in target_keys and value.shape == target[key].shape
        }
        score = len(compatible)
        if best is None or score > best[0]:
            best = (score, prefix, compatible)

    score, prefix, weights = best
    missing = sorted(target_keys - set(weights.keys()))
    if missing:
        raise RuntimeError(
            f"\nBACKBONE LOADING FAILED: {repo_id}\n"
            f"Matched: {score}/{len(target_keys)}\n"
            f"Prefix: {prefix}\n"
            f"Missing: {missing[:20]}"
        )

    model.load_state_dict(weights, strict=True)
    model.eval().to(DEVICE)
    feature_dim = 512 if architecture == "resnet18" else 2048
    print(f"\nSuccessfully loaded!")
    print(f"Prefix: {prefix}")
    print(f"Backbone tensors: {score}/{len(target_keys)}")
    print(f"Feature dimension: {feature_dim}")
    return model


SSL_BACKBONES = {}
for name, config in SSL_MODELS.items():
    try:
        SSL_BACKBONES[name] = load_ssl_model(config)
    except Exception as e:
        print(f"\nWARNING: could not load {name}: {e}")
        print(f"Skipping {name}.")
EMB={}
for name, backbone in SSL_BACKBONES.items():
    key = f"{name} (external SSL)"
    EMB[key] = embed_backbone(backbone, X_db)
    print(f"{key}: embeddings extracted -> {EMB[key].shape}")

print("\nAvailable embeddings:", list(EMB.keys()))

## Part 2 — Retrieval metrics

In [ ]:
# [Part 2] Metrics
def l2_normalize(Z, eps=1e-12):
    norms = np.linalg.norm(Z, axis=1, keepdims=True)
    return Z / np.clip(norms, eps, None)

def relevance_vector(ids, y_db, q):
    ids = np.asarray(ids)
    ids = ids[ids >= 0]          # drop padding
    ids = ids[ids != q]          # drop the query itself
    return y_db[ids] == y_db[q]

def query_metrics(rel, n_rel):
    rel = np.asarray(rel, dtype=bool)
    p10 = rel[:10].sum() / 10 if len(rel) >= 10 else rel.sum() / max(len(rel), 1)
    r50 = rel[:50].sum() / max(n_rel, 1)
    if n_rel == 0:
        return p10, r50, 0.0
    hits = np.where(rel)[0] + 1
    precisions = np.array([rel[:h].sum() / h for h in hits])
    ap = precisions.mean() if len(precisions) else 0.0
    return p10, r50, ap

def per_query_metrics(emb, y_db, q_pos):
    E = l2_normalize(emb)
    mask = np.ones(len(y_db), dtype=bool)
    mask[q_pos] = False
    E_clean = E[mask]
    y_clean = y_db[mask]
    P10, R50, AP, MS = [], [], [], []
    for q in q_pos:
        t0 = time.perf_counter()
        sims = E_clean @ E[q]
        dt = (time.perf_counter() - t0) * 1000
        ranking = np.argsort(-sims)
        rel = (y_clean[ranking] == y_db[q])
        n_rel = int((y_clean == y_db[q]).sum())
        p10, r50, ap = query_metrics(rel, n_rel)
        P10.append(p10); R50.append(r50); AP.append(ap); MS.append(dt)
    return {"P10": np.array(P10), "R50": np.array(R50),
            "AP": np.array(AP), "ms": np.array(MS)}

def evaluate_retrieval(emb, y_db, q_pos):
    m = per_query_metrics(emb, y_db, q_pos)
    return {"Dim": emb.shape[1], "mAP": m["AP"].mean(),
            "P@10": m["P10"].mean(), "R@50": m["R50"].mean(),
            "ms/req": m["ms"].mean()}

# Sanity check: a perfect ranking must give AP = P@10 = 1
q0 = q_pos[0]
perfect = np.argsort(-(y_db == y_db[q0]).astype(float), kind="stable")
print("sanity:", query_metrics(relevance_vector(perfect, y_db, q0),
                                (y_db == y_db[q0]).sum() - 1))

def results_table():
    df = pd.DataFrame(
        {n: evaluate_retrieval(EMB[n], y_db, q_pos) for n in EMB}
    ).T
    df["Dim"] = df["Dim"].astype(int)
    return df.round(3)

results_table()

## Part 3 — SSL fine-tuning (optional, GPU recommended)

In [ ]:
# [Part 3.0] Shared blocks
norm = transforms.Normalize(MEAN, STD)
simclr_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=5),
    transforms.ToTensor(), norm,
])
byol_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.3, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), norm,
])

class TwoViews(Dataset):
    def __init__(self, X, tf): self.X, self.tf = X, tf
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        img = Image.fromarray(self.X[i])
        return self.tf(img), self.tf(img)

def make_loader(tf):
    return DataLoader(TwoViews(X_db, tf), batch_size=128,
                      shuffle=True, drop_last=True,
                      num_workers=NUM_WORKERS)

class ProjectionHead(nn.Module):
    def __init__(self, dim=512, hid=256, dout=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hid),
                                 nn.ReLU(inplace=True),
                                 nn.Linear(hid, dout))
    def forward(self, h): return self.net(h)

def weight_distance(a, b):
    with torch.no_grad():
        return float(torch.sqrt(sum(((pa - pb) ** 2).sum()
                   for pa, pb in zip(a.parameters(), b.parameters()))))

In [ ]:
# [Part 3.1] SimCLR
def nt_xent(z1, z2, tau=0.5):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)
    sim = z @ z.T / tau
    mask = torch.eye(2 * N, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)
    targets = torch.cat([torch.arange(N, 2 * N),
                         torch.arange(0, N)]).to(z.device)
    return F.cross_entropy(sim, targets)

def fine_tune_simclr(encoder, epochs):
    enc = copy.deepcopy(encoder).to(DEVICE)
    head = ProjectionHead().to(DEVICE)
    opt = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()),
                           lr=1e-4, weight_decay=1e-6)
    loader = make_loader(simclr_tf)
    enc.train()
    for ep in range(epochs):
        total, n = 0.0, 0
        for v1, v2 in loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            z1 = head(enc(v1)); z2 = head(enc(v2))
            loss = nt_xent(z1, z2)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); n += 1
        print(f"[SimCLR] epoch {ep+1}/{epochs} loss={total/n:.4f}")
    enc.eval()
    return enc

if FINE_TUNE:
    enc_simclr_ft = fine_tune_simclr(enc_sup, EPOCHS)
    EMB["SimCLR (ImageNet + FT)"] = embed_backbone(enc_simclr_ft, X_db)
    print("weight distance to init:",
          weight_distance(enc_simclr_ft, enc_sup))

In [ ]:
# [Part 3.2] MoCo
class MoCoLite(nn.Module):
    def __init__(self, base_encoder, dim=512, dout=64,
                 K=1024, m=0.999, tau=0.2):
        super().__init__()
        self.enc_q = copy.deepcopy(base_encoder).to(DEVICE)
        self.proj_q = ProjectionHead(dim, 256, dout).to(DEVICE)
        self.enc_k = copy.deepcopy(base_encoder).to(DEVICE)
        self.proj_k = copy.deepcopy(self.proj_q).to(DEVICE)
        for p in list(self.enc_k.parameters()) + list(self.proj_k.parameters()):
            p.requires_grad = False
        self.register_buffer("queue", F.normalize(torch.randn(dout, K), dim=0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))
        self.m = m
        self.tau = tau

    @torch.no_grad()
    def momentum_update(self):
        for pq, pk in zip(self.enc_q.parameters(), self.enc_k.parameters()):
            pk.data.mul_(self.m).add_(pq.data, alpha=1 - self.m)
        for pq, pk in zip(self.proj_q.parameters(), self.proj_k.parameters()):
            pk.data.mul_(self.m).add_(pq.data, alpha=1 - self.m)

    @torch.no_grad()
    def enqueue(self, keys):
        keys = F.normalize(keys, dim=1).T
        bs = keys.size(1)
        ptr = int(self.queue_ptr)
        end = ptr + bs
        if end <= self.queue.size(1):
            self.queue[:, ptr:end] = keys
        else:
            self.queue[:, ptr:] = keys[:, :self.queue.size(1) - ptr]
            self.queue[:, :end - self.queue.size(1)] = keys[:, self.queue.size(1) - ptr:]
        self.queue_ptr[0] = (ptr + bs) % self.queue.size(1)

    def forward(self, v_q, v_k):
        z_q = F.normalize(self.proj_q(self.enc_q(v_q)), dim=1)
        with torch.no_grad():
            z_k = F.normalize(self.proj_k(self.enc_k(v_k)), dim=1)
        return z_q, z_k

def info_nce_queue(z_q, z_k, queue, tau=0.2):
    l_pos = (z_q * z_k).sum(dim=1, keepdim=True)
    l_neg_q = z_q @ queue
    l_neg_batch = z_q @ z_k.T
    N = z_q.size(0)
    eye = torch.eye(N, device=z_q.device, dtype=torch.bool)
    l_neg_batch = l_neg_batch.masked_fill(eye, -1e9)
    logits = torch.cat([l_pos, l_neg_q, l_neg_batch], dim=1) / tau
    labels = torch.zeros(z_q.size(0), dtype=torch.long, device=z_q.device)
    return F.cross_entropy(logits, labels)

def fine_tune_moco(encoder, epochs):
    model = MoCoLite(encoder).to(DEVICE)
    model.enc_q.train()
    opt = torch.optim.Adam(
        list(model.enc_q.parameters()) + list(model.proj_q.parameters()),
        lr=1e-4)
    loader = make_loader(simclr_tf)
    for ep in range(epochs):
        total, n = 0.0, 0
        for v1, v2 in loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            z_q, z_k = model(v1, v2)
            loss = info_nce_queue(z_q, z_k, model.queue, tau=model.tau)
            opt.zero_grad(); loss.backward(); opt.step()
            model.momentum_update()
            model.enqueue(z_k)
            total += loss.item(); n += 1
        print(f"[MoCo] epoch {ep+1}/{epochs} loss={total/n:.4f}")
    model.enc_q.eval()
    return model.enc_q

if FINE_TUNE:
    enc_moco_ft = fine_tune_moco(enc_sup, EPOCHS)
    EMB["MoCo (ImageNet + FT)"] = embed_backbone(enc_moco_ft, X_db)
    print("weight distance to init:",
          weight_distance(enc_moco_ft, enc_sup))

In [ ]:
# [Part 3.3] BYOL
class BYOLProjection(nn.Module):
    def __init__(self, dim=512, hid=256, dout=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hid),
                                 nn.BatchNorm1d(hid),
                                 nn.ReLU(inplace=True),
                                 nn.Linear(hid, dout))
    def forward(self, h): return self.net(h)

class BYOLPredictor(nn.Module):
    def __init__(self, dim=256, hid=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hid),
                                 nn.BatchNorm1d(hid),
                                 nn.ReLU(inplace=True),
                                 nn.Linear(hid, dim))
    def forward(self, z): return self.net(z)

def byol_loss(p1, p2, z1, z2):
    p1 = F.normalize(p1, dim=1); p2 = F.normalize(p2, dim=1)
    z1 = F.normalize(z1, dim=1).detach()
    z2 = F.normalize(z2, dim=1).detach()
    return 2 - 2 * (p1 * z2).sum(dim=1).mean() + 2 - 2 * (p2 * z1).sum(dim=1).mean()

@torch.no_grad()
def ema_update(target, online, m=0.996):
    for pt, po in zip(target.parameters(), online.parameters()):
        pt.data.mul_(m).add_(po.data, alpha=1 - m)

def fine_tune_byol(encoder, epochs):
    online = copy.deepcopy(encoder).to(DEVICE).train()
    target = copy.deepcopy(encoder).to(DEVICE).train()
    proj_on = BYOLProjection().to(DEVICE)
    proj_tg = copy.deepcopy(proj_on).to(DEVICE)
    pred = BYOLPredictor().to(DEVICE)
    for p in list(target.parameters()) + list(proj_tg.parameters()):
        p.requires_grad = False
    opt = torch.optim.Adam(
        list(online.parameters()) + list(proj_on.parameters()) + list(pred.parameters()),
        lr=1e-4)
    loader = make_loader(byol_tf)
    for ep in range(epochs):
        total, n = 0.0, 0
        for v1, v2 in loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            with torch.no_grad():
                t1 = proj_tg(target(v1)); t2 = proj_tg(target(v2))
            p1 = pred(proj_on(online(v1))); p2 = pred(proj_on(online(v2)))
            loss = byol_loss(p1, p2, t1, t2)
            opt.zero_grad(); loss.backward(); opt.step()
            ema_update(target, online)
            ema_update(proj_tg, proj_on)
            total += loss.item(); n += 1
        print(f"[BYOL] epoch {ep+1}/{epochs} loss={total/n:.4f}")
    target.eval()
    return target

if FINE_TUNE:
    enc_byol_ft = fine_tune_byol(enc_sup, EPOCHS)
    EMB["BYOL (ImageNet + FT)"] = embed_backbone(enc_byol_ft, X_db)
    print("weight distance to init:",
          weight_distance(enc_byol_ft, enc_sup))

## Part 4 — Comparing all representation spaces

In [ ]:
# [Part 4.1] Global table
table = results_table()
display(table)
best_name = table["mAP"].astype(float).idxmax()
print("Best representation:", best_name)

In [ ]:
# [Part 4.2] Per-class analysis
m  = per_query_metrics(EMB[best_name], y_db, q_pos)
yq = y_db[q_pos]
rows = []
for c, name in enumerate(CLASS_NAMES):
    s = yq == c
    rows.append({"class": name, "#queries": int(s.sum()),
                 "P@10":    m["P10"][s].mean() if s.any() else np.nan,
                 "R@50":    m["R50"][s].mean() if s.any() else np.nan,
                 "mean AP": m["AP"][s].mean()  if s.any() else np.nan})
pd.DataFrame(rows).round(3)

In [ ]:
# [Part 4.3] Confusion pair + visual grid
def confusion_pairs(emb, y_db, q_pos, k=10):
    C = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    E = l2_normalize(emb)
    mask = np.ones(len(y_db), dtype=bool)
    mask[q_pos] = False
    E_clean = E[mask]; y_clean = y_db[mask]
    for q in q_pos:
        sims = E_clean @ E[q]
        top = np.argsort(-sims)[:k]
        for t in top:
            C[y_db[q], y_clean[t]] += 1
    return C

C = confusion_pairs(EMB[best_name], y_db, q_pos)
off = C + C.T; np.fill_diagonal(off, 0)
a, b = np.unravel_index(off.argmax(), off.shape)
print(pd.DataFrame(C, index=CLASS_NAMES, columns=CLASS_NAMES))
print(f"Most confused pair: {CLASS_NAMES[a]} <-> {CLASS_NAMES[b]}")

def show_results(emb, q, k=8):
    E = l2_normalize(emb)
    mask = np.ones(len(y_db), dtype=bool)
    mask[q_pos] = False
    E_clean = E[mask]; y_clean = y_db[mask]
    sims = E_clean @ E[q]
    top = np.argsort(-sims)[:k]
    idx_all = np.where(mask)[0]
    fig, axes = plt.subplots(1, k + 1, figsize=(1.6 * (k + 1), 1.8))
    axes[0].imshow(X_db[q]); axes[0].set_title("query", fontsize=8)
    axes[0].axis("off")
    for ax, t in zip(axes[1:], top):
        gi = idx_all[t]
        ax.imshow(X_db[gi])
        color = "green" if y_db[gi] == y_db[q] else "red"
        ax.set_title(CLASS_NAMES[y_db[gi]], fontsize=7, color=color)
        ax.axis("off")
    plt.tight_layout(); plt.show()

for q in [q for q in q_pos if y_db[q] in (a, b)][:3]:
    show_results(EMB[best_name], q)

In [ ]:
# [Part 4.4] t-SNE of every space
from sklearn.manifold import TSNE
def plot_tsne(emb, y, title, ax, n_max=800):
    E = l2_normalize(emb)
    idx = np.random.default_rng(0).choice(len(E), size=min(n_max, len(E)), replace=False)
    z = TSNE(n_components=2, perplexity=30, init="pca",
             random_state=0).fit_transform(E[idx])
    sc = ax.scatter(z[:, 0], z[:, 1], c=y[idx], cmap="tab10", s=6)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])

ncols = 3; nrows = int(np.ceil(len(EMB) / ncols))
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(5 * ncols, 4.4 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, name in zip(axes, EMB):
    plot_tsne(EMB[name], y_db, name, ax)
for ax in axes[len(EMB):]:
    ax.axis("off")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

## Part 5 — FAISS trade-off

In [ ]:
# [Part 5] Exact vs IVF
import faiss
faiss.omp_set_num_threads(1)

xb = np.ascontiguousarray(l2_normalize(EMB[best_name]).astype("float32"))
n_db, d = xb.shape
Q = np.ascontiguousarray(xb[q_pos])
nlist = int(round(np.sqrt(n_db)))
print(f"Indexing space: {best_name} | N={n_db}, d={d}, nlist={nlist}")

def drop_self(I, k):
    out = np.empty((I.shape[0], k), dtype=I.dtype)
    for i, row in enumerate(I):
        row = row[row != q_pos[i]]
        row = row[row >= 0]
        out[i, :min(k, len(row))] = row[:min(k, len(row))]
        if len(row) < k:
            out[i, len(row):] = -1
    return out

def bench(index, k=10, reps=5):
    index.search(Q[:1], k)  # warm-up
    t0 = time.perf_counter()
    for _ in range(reps):
        index.search(Q, k)
    return (time.perf_counter() - t0) / reps / len(Q) * 1000

def full_map(index):
    _, I = index.search(Q, n_db)
    I = drop_self(I, n_db - 1)
    aps = []
    for i, row in enumerate(I):
        row = row[row >= 0]
        rel = (y_db[row] == y_db[q_pos[i]])
        n_rel = int((y_db == y_db[q_pos[i]]).sum()) - 1
        if n_rel == 0 or rel.sum() == 0:
            aps.append(0.0); continue
        hits = np.where(rel)[0] + 1
        prec = np.array([rel[:h].sum() / h for h in hits])
        aps.append(float(prec.mean()))
    return float(np.mean(aps))

# Exact baseline
index_flat = faiss.IndexFlatIP(d)
index_flat.add(xb)
ms_flat = bench(index_flat)
map_flat = full_map(index_flat)
print(f"Exact: mAP={map_flat:.4f}, {ms_flat:.3f} ms/query")

# IVF sweep
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(xb)
index_ivf.add(xb)

rows = []
for nprobe in [1, 2, 5, 10, 20, 50, nlist]:
    index_ivf.nprobe = nprobe
    ms = bench(index_ivf)
    map_ = full_map(index_ivf)
    # recall@10 vs exact
    _, Ie = index_flat.search(Q, 10)
    _, Ia = index_ivf.search(Q, 10)
    Ie = drop_self(Ie, 10); Ia = drop_self(Ia, 10)
    hits = sum(len(set(a[a >= 0]) & set(e[e >= 0])) for a, e in zip(Ia, Ie))
    recall = hits / (10 * len(Q))
    rows.append({"nprobe": nprobe, "mAP": map_, "ms/req": ms,
                 "Recall@10 vs exact": recall})

ivf_table = pd.DataFrame(rows).round(4)
display(ivf_table)
assert abs(ivf_table.iloc[-1]["Recall@10 vs exact"] - 1.0) < 1e-6, \
    "nprobe = nlist should recover the exact ranking"

# Trade-off curve
fig, ax1 = plt.subplots(figsize=(6, 4))
ax1.plot(ivf_table["nprobe"], ivf_table["mAP"], "o-", color="C0", label="mAP")
ax1.set_xlabel("nprobe"); ax1.set_ylabel("mAP", color="C0")
ax1.tick_params(axis="y", labelcolor="C0")
ax2 = ax1.twinx()
ax2.plot(ivf_table["nprobe"], ivf_table["ms/req"], "s--", color="C1", label="ms/req")
ax2.set_ylabel("ms/req", color="C1")
ax2.tick_params(axis="y", labelcolor="C1")
plt.title(f"FAISS IVF trade-off ({best_name})")
plt.tight_layout(); plt.show()

## Answers to the six questions (handout)

**Q1.** (mAP comparison) ...

**Q2.** (before/after fine-tuning) ...

**Q3.** (color invariance) ...

**Q4.** (most confused pair) ...

**Q5.** (FAISS operating point) ...

**Q6.** (cosine vs Euclidean) ...

**Synthesis (10–15 lines, red thread).** ...